# 🚀 Имитационное обучение: GAIL для F‑16 (линейная продольная модель)

Лаконичный, удобный для чтения на GitHub пример GAIL (Generative Adversarial Imitation Learning) для продольной динамики F‑16: формирование демонстраций эксперта, обучение GAIL и быстрая визуальная проверка качества слежения по углу атаки.


## 📋 Оглавление
- [📖 Введение](#📖-Введение)
- [📦 Импорты](#📦-Импорты)
- [🔧 Параметры эксперимента](#🔧-Параметры-эксперимента)
- [✈️ Инициализация среды F-16](#✈️-Инициализация-среды-F-16)
- [🎓 Экспертные демонстрации (PD)](#🎓-Экспертные-демонстрации-(PD))
- [🤖 Обучение GAIL](#🤖-Обучение-GAIL)
- [📈 Визуализация](#📈-Визуализация)
- [💡 Советы](#💡-Советы)
- [📄 Лицензия](#📄-Лицензия)


## 📖 Введение
GAIL обучает политику повторять поведение эксперта, используя состязательное обучение с дискриминатором. В этом примере эксперт — простой PD‑регулятор по углу атаки и угловой скорости тангажа. Затем мы обучаем GAIL и проверяем слежение по α.



In [ ]:
# 📦 Импорты
import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

from tensoraerospace.agent.gail.model import GAIL
from tensoraerospace.agent.pid import PID
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 🔧 Параметры эксперимента
np.random.seed(42)

dt = 0.01  # Дискретизация
tp = generate_time_period(tn=20, dt=dt)  # Временной период
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)  # Количество временных шагов
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=1000, output_rad=True), [1, -1]
)  # Заданный сигнал

2024-07-30 11:00:19.740289: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-30 11:00:19.740320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-30 11:00:19.740971: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-30 11:00:19.745397: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-30 11:00:20.444520: W tensorflow/compiler/tf2

-133.5645


In [ ]:
# ✈️ Инициализация среды F-16
# initial_state соответствует выбранным состояниям
initial_state = [[0], [0], [0]]  # theta, alpha, q

env = gym.make(
    'LinearLongitudinalF16-v0',
    number_time_steps=number_time_steps,
    initial_state=initial_state,
    reference_signal=reference_signals,
    use_reward=False,
    state_space=["theta", "alpha", "q"],
    output_space=["theta", "alpha", "q"],
    control_space=["ele"],
    tracking_states=["alpha"],
)

env.reset()


In [ ]:
# 🎓 Экспертные демонстрации (PD)
kp, kd = 6.0, 1.0

expert_traj = []  # список из [state, action]
xt, info = env.reset()

for step in range(number_time_steps - 2):
    setpoint = reference_signals[0, step]
    alpha = float(xt[1, 0])  # alpha во 2-й компоненте
    q_rate = float(xt[2, 0])

    u = -kp * (alpha - setpoint) - kd * q_rate
    ut = np.array([[u]], dtype=np.float32)

    expert_traj.append(np.hstack([xt.reshape(-1), ut.reshape(-1)]))

    xt, reward, terminated, truncated, info = env.step(ut)
    if terminated or truncated:
        break

expert_data = np.array(expert_traj, dtype=np.float32)
expert_data.shape


In [ ]:
# 🤖 Обучение GAIL
learning_rate = 3e-3
max_steps_per_update = 20
mini_batch_size = 16
ppo_epochs = 4

agent = GAIL(
    env=env,
    learning_rate=learning_rate,
    max_steps=max_steps_per_update,
    mini_batch_size=mini_batch_size,
    epochs=ppo_epochs,
    data=expert_data,
)
agent.learn(max_frames=5000, max_reward=-1)


In [ ]:
# 📈 Визуализация
xt, _ = env.reset()
traj_alpha = []
traj_theta = []
traj_q = []
traj_u = []

for step in range(number_time_steps - 2):
    state_t = torch.from_numpy(xt.reshape(1, -1)).float().to(device)
    with torch.no_grad():
        dist, _ = agent.model(state_t)
        ut_tensor = dist.sample()
        ut = ut_tensor.detach().cpu().numpy()

    xt, _, terminated, truncated, _ = env.step(ut.reshape(1, 1))

    traj_theta.append(float(xt[0, 0]))
    traj_alpha.append(float(xt[1, 0]))
    traj_q.append(float(xt[2, 0]))
    traj_u.append(float(ut.reshape(-1)[0]))

    if terminated or truncated:
        break

fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
axes[0].plot(tps[: len(traj_alpha)], np.rad2deg(np.array(traj_alpha)), label="alpha [deg]")
axes[0].plot(tps[: len(traj_alpha)], np.rad2deg(reference_signals[0, : len(traj_alpha)]), '--', label="ref [deg]")
axes[0].set_ylabel("alpha, deg")
axes[0].legend()

axes[1].plot(tps[: len(traj_theta)], np.rad2deg(np.array(traj_theta)), label="theta [deg]")
axes[1].set_ylabel("theta, deg")
axes[1].legend()

axes[2].plot(tps[: len(traj_q)], np.rad2deg(np.array(traj_q)), label="q [deg/s]")
axes[2].plot(tps[: len(traj_u)], np.rad2deg(np.array(traj_u)), label="u [deg]")
axes[2].set_ylabel("q / u")
axes[2].set_xlabel("time, s")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# (необязательно) Альтернативная компактная визуализация
# Дублирует основную идею, но оставляем один вариант для краткости стиля
pass

## 💡 Советы
- **Сходимость**: уменьшите `learning_rate` или увеличьте `mini_batch_size`, если поведение шумное
- **Демонстрации**: замените PD на `PID` для более качественного эксперта
- **Повторяемость**: зафиксируйте `np.random.seed(...)` и (опционально) сиды для PyTorch

## 📄 Лицензия
Этот пример распространяется на условиях лицензии проекта (см. LICENSE в корне репозитория).